# Alice/Bob без пошагового барьера
Qwen3-4B без thinking, 1024 токена на агента.
Независимые циклы; эксклюзивный доступ к кешу только при публикации.
Kernel: **AsyncLLM (uv)**.


In [ ]:
import asyncio
import gc
import os
import re
import sys
import time
from contextlib import aclosing
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().parents[1] if Path.cwd().name == "parallel_thinking" else Path.cwd()
os.chdir(ROOT)
for path in (ROOT / "python", ROOT / "scripts/async_thoughts", ROOT / "scripts/parallel_thinking"):
    sys.path.insert(0, str(path))
os.environ.update(
    CUDA_DEVICE_ORDER="PCI_BUS_ID",
    CUDA_VISIBLE_DEVICES="1",
    TVM_FFI_CUDA_ARCH_LIST="8.6",
    OMP_NUM_THREADS="8",
)
os.environ["PATH"] = f"{Path(sys.executable).parent}:/usr/local/cuda/bin:" + os.environ["PATH"]

import torch
from async_helpers import PublicationGate, isolated, one_token, recorded_example, together
from async_thoughts.training import DemoRecorder
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay

TEACHER = "models/Qwen3-4B"
WORKERS = ("Alice", "Bob")
PROBLEM = (
    "Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and "
    "$R=(11,3,9)$ . What is the surface area of the cube?"
)

In [ ]:
async def solve_no_barrier(model, *, max_steps=1024, prompt_ids=None, bob_delay=0.0):
    started = time.perf_counter()
    llm = AsyncLLM(
        str(model),
        dtype=torch.bfloat16,
        memory_ratio=0.65,
        max_running_req=4,
        max_seq_len_override=16384,
        cuda_graph_bs=[1, 2],
        cuda_graph_max_bs=2,
        shared_cuda_graph_max_depth=6,
        shared_cuda_graph_prefill_rows=[32, 256],
        max_prefill_rows=256,
        distributed_addr="tcp://127.0.0.1:0",
    )
    llm = DemoRecorder(llm)
    blocks = []
    try:
        tokenizer = llm.tokenizer
        encode = lambda text: tokenizer.encode(text, add_special_tokens=False)

        fmt = MathFormatting(
            tokenizer, WORKERS, enable_thinking=False
        )
        for _ in range(6):
            blocks.append(await llm.create_block())
        common, others_header, own_header, alice, bob, answer_block = blocks
        worker_blocks = [alice, bob]
        contexts = [
            AsyncContext(cache_view=[common, others_header, bob, own_header, alice]),
            AsyncContext(cache_view=[common, others_header, alice, own_header, bob]),
        ]
        initial_ids = (
            list(prompt_ids) if prompt_ids is not None else encode(fmt.apply_chat_template(PROBLEM))
        )

        await llm.forward(initial_ids, write_to=common, return_logits=False)
        await together(
            llm.forward(
                encode(fmt.current_step_header), write_to=others_header, return_logits=False
            ),
            llm.forward(
                encode(fmt.current_worker_header), write_to=own_header, return_logits=False
            ),
        )

        prefixes = [
            f"{fmt.get_step_prefix(WORKERS[0], 1)}Hi, I'm {WORKERS[0]}. Here's how we can collaborate",
            f"{fmt.get_step_prefix(WORKERS[1], 1)}Hi, I'm {WORKERS[1]}.",
        ]
        current = [encode(prefix) for prefix in prefixes]

        await together(
            *(
                llm.forward(
                    ids[:-1],
                    cache_view=[common, others_header, own_header, block],
                    return_logits=False,
                )
                for ids, block in zip(current, worker_blocks)
            )
        )
        for ctx, ids in zip(contexts, current):
            ctx.next_input_id = ids[-1]

        history, events = [], []
        step_indices, generated = [1, 1], [0, 0]
        sampled = [[], []]
        forbidden = [
            tokenizer.vocab[t]
            for t in ("<|im_start|>", "<|endoftext|>")
            if t in tokenizer.vocab
        ]

        eos = tokenizer.eos_token_id
        forbidden = [token for token in forbidden if token != eos]
        generation_started = time.perf_counter()

        gate = PublicationGate()
        stopped = False
        progress = []

        async def generate_agent(i):
            nonlocal stopped
            ctx = contexts[i]
            for local_step in range(1, max_steps + 1):
                if stopped:
                    break

                if i == 1 and bob_delay and local_step % 32 == 0:
                    await asyncio.sleep(bob_delay)
                async with gate.forward():
                    token = await one_token(llm, ctx, forbidden)
                    current[i].append(token)
                    generated[i] += 1
                    sampled[i].append(token)
                    progress.append(tuple(generated))
                if token == eos:
                    stopped = True
                    break
                if fmt.is_end_of_step(current[i]):

                    async with gate.publish():
                        await llm.forward(cache_view=ctx, return_logits=False)
                        block = worker_blocks[i]
                        assert block.token_ids == current[i]
                        await llm.append_block(common, block)
                        history.extend(current[i])
                        assert common.token_ids == initial_ids + history
                        events.append(
                            {
                                "worker": WORKERS[i],
                                "step": step_indices[i],
                                "tokens": len(current[i]),
                                "generated": tuple(generated),
                                "text": tokenizer.decode(current[i]),
                            }
                        )
                        await llm.free_block(block)
                        step_indices[i] += 1
                        current[i] = encode(fmt.get_step_prefix(WORKERS[i], step_indices[i]))
                        await llm.forward(current[i][:-1], cache_view=ctx, return_logits=False)
                        ctx.next_input_id = current[i][-1]
                assert ctx.output_block.token_ids == current[i][:-1]
                assert ctx.next_input_id == current[i][-1]

        await together(generate_agent(0), generate_agent(1))
        iteration = max(generated)

        await together(*(llm.forward(cache_view=ctx, return_logits=False) for ctx in contexts))
        for i, ctx in enumerate(contexts):
            assert ctx.output_block.token_ids == current[i]
        parallel_seconds = time.perf_counter() - generation_started
        transcript = fmt.history_header + fmt.sep + tokenizer.decode(history)
        transcript += fmt.sep + fmt.current_step_header
        transcript += "".join(
            tokenizer.decode(ids) + fmt.incomplete_step + fmt.sep for ids in current
        )

        suffix = fmt.s1_finisher_suffix
        answer_ids = encode(suffix)
        answer_ctx = AsyncContext(
            cache_view=[common, others_header, alice, own_header, bob, answer_block]
        )
        await llm.forward(answer_ids[:-1], cache_view=answer_ctx, return_logits=False)
        answer_ctx.next_input_id = answer_ids[-1]
        answer_tokens = []
        async with aclosing(
            llm.async_generate(
                answer_ctx,
                max_steps=64,
                forbid_ids=forbidden,
                sampling_params=SamplingParams(temperature=0.0, max_tokens=64),
            )
        ) as stream:
            async for token in stream:
                if token == eos:
                    break
                answer_tokens.append(token)
                if fmt.get_final_answer(suffix + tokenizer.decode(answer_tokens)) is not None:
                    break
        answer_text = suffix + tokenizer.decode(answer_tokens)
        result = {
            "model": str(model),
            "problem": PROBLEM,
            "rounds": iteration,
            "generated_tokens": dict(zip(WORKERS, generated)),
            "completed_steps": len(events),
            "merge_events": events,
            "transcript": transcript,
            "answer_text": answer_text,
            "answer": fmt.get_final_answer(answer_text),
            "parallel_seconds": parallel_seconds,
            "total_seconds": time.perf_counter() - started,
        }
        result["generated_text"] = {w: tokenizer.decode(ids) for w, ids in zip(WORKERS, sampled)}
        result["prompt_ids"] = initial_ids
        result["history_ids"] = history
        result["current_ids"] = current
        result["answer_ids"] = answer_tokens
        result["progress"] = progress
        result["max_token_gap"] = max(abs(a - b) for a, b in progress)
        result["example"] = recorded_example(llm)
        return result
    finally:

        try:
            for block in blocks:
                await llm.free_block(block)
        finally:
            await llm.close()
            gc.collect()
            torch.cuda.empty_cache()

def show_result(result):
    print("Answer:", result["answer"],
          "Published blocks:", result["completed_steps"])
    print("Generated:", result["generated_tokens"], "Maximum token gap:", result["max_token_gap"])
    print("Generated role headers:", {
        w: re.findall(r"\*\*(?:Alice|Bob) \[\d+\]:\*\*", text)
        for w, text in result["generated_text"].items()
    })
    print("Generated think tags:", {
        w: re.findall(r"</?think>", text) for w, text in result["generated_text"].items()
    })
    display(Markdown(result["transcript"] + "\n\n" + result["answer_text"]))


In [ ]:
normal = isolated(solve_no_barrier, TEACHER)
show_result(normal)


## Независимый темп
Задержка Bob на 50 мс каждые 32 токена.


In [ ]:
delayed = isolated(solve_no_barrier, TEACHER, bob_delay=0.05)
print("Same trajectory:", delayed["history_ids"] == normal["history_ids"])
assert delayed["max_token_gap"] > 1

for result in (normal, delayed):
    example = result["example"]
    replay = Replay.from_trace(
        example["trace"], query_chunk_size=64, attention_checkpointing=True, cache_plans=True
    )
    print(
        "Trace rows:",
        example["trace"].num_tokens,
        "Targets:",
        sum(t != -100 for t in example["targets"]),
    )
show_result(delayed)